In [ ]:
df_reviews = spark.table("Enriched.Reviews")
display(df_reviews.limit(10))

## Get Sentiment and Sentiment Score with AI Function

In [ ]:
from synapse.ml.spark.aifunc.DataFrameExtensions import AIFunctions

responses = df_reviews.ai.generate_response(
    prompt = "Return sentiment and sentiment score from user\'s review {Review} of wines. Review text is in Czech language. Return Positive, Negative, Neutral and its score between -1 to 1. If you are not able detect sentiment, return null. Return JSON with Schema: sentiment:string, score:double"
   # prompt = "Return sentiment and sentiment score from user\'s review {Review} of wines. Review text is in Czech language. Return Positive, Negative, Neutral and its score between -1 to 1. If you are not able detect sentiment, return null"
  , is_prompt_template=True
  , output_col="response"
  , response_format="json_object"
  )
display(responses.limit(10))


In [ ]:
from pyspark.sql.functions import get_json_object, col
from pyspark.sql.types import DecimalType

df_reviews = responses.select(
    "WineCode", "Date", "Review", "ReviewId", "User",
    
    # Sentiment from JSON string
    get_json_object(col("response"), "$.sentiment").alias("Sentiment"),
    
    # Score from JSON string
    get_json_object(col("response"), "$.score").cast(DecimalType(5,2)).alias("Score")
)
display(df_reviews.limit(10))

## Get Meal for Wine with AI Function

In [ ]:
from synapse.ml.spark.aifunc.DataFrameExtensions import AIFunctions

responses = df_reviews.ai.generate_response(
    prompt = "Return name of meal which is mentioned in user\'s review {Review} of wines. Review text is in Czech language. Return only name of food in Czech without quote. Do not return name or type of wine such as Pálava, Rulandské šedé. If there is not any meal, return null."    
  , is_prompt_template=True
  , output_col="Wine2Meal"
  , response_format="text"
  )
display(responses.limit(10))


In [ ]:
df_reviews = responses.select(
    "WineCode", "Date", "Review", "ReviewId", "User", "Sentiment", "Score", "Wine2Meal"
    
)
display(df_reviews.limit(10))

## Get Name, Surname and Domain with AI Function

In [ ]:
from synapse.ml.spark.aifunc.DataFrameExtensions import AIFunctions

entities = df_reviews.ai.extract(labels=["Name", "Surname", "Domain"], input_col="User")
display(entities.limit(10))

In [ ]:
from pyspark.sql.functions import element_at

df_selected = entities.select(
    "ReviewId",
    "Date",
    "WineCode",
    "Review",
    "Sentiment",
    "Score",
    "Wine2Meal",
    "User",
    element_at(col("Name"), 1).alias("UserName"),     # index 1 = první prvek
    element_at(col("Surname"), 1).alias("UserSurname"),
    element_at(col("Domain"), 1).alias("UserDomain")    
).drop("Name", "Surname", "Domain","generate_response_error","User_extract_error")  # Odstraní původní array sloupce

display(df_selected.limit(10))

## Review Dimension

In [ ]:
spark.conf.get('spark.sql.parquet.vorder.default')
spark.conf.set('spark.sql.parquet.vorder.default', 'true')
spark.conf.get('spark.sql.parquet.vorder.default')

In [ ]:
# Write 
df_selected.write.format("delta").mode("overwrite").option("mergeSchema", "true").saveAsTable("Curated.Review_Dimension")


print(f"Written {df_selected.count()} rows.")